In [1]:
import os
import glob
from pathlib import Path
import pandas as pd
import dataframe_image as dfi
from fpdf import FPDF
import pdfkit
import math
import pyreadstat




In [2]:
RAW_PATH = Path("../data/raw")
CLEANED_PATH = Path("../data/cleaned")
CLEANED_PATH.mkdir(parents=True, exist_ok=True)


In [3]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df.dropna(axis=1, how='all', inplace=True)
    df.drop_duplicates(inplace=True)
    
    # Replace NaNs with empty strings
    df.fillna("", inplace=True)
    return df


def save_df_to_pdf(df: pd.DataFrame, pdf_path: Path):
    html_path = pdf_path.with_suffix(".html")
    df.head(50).to_html(html_path, index=False)

    # Adjust path as needed
    path_wkhtmltopdf = r"C:\Program Files\wkhtmltopdf\bin\wkhtmltopdf.exe"
    config = pdfkit.configuration(wkhtmltopdf=path_wkhtmltopdf)

    pdfkit.from_file(str(html_path), str(pdf_path), configuration=config)


In [4]:
# Define this outside the loop
def save_df_to_pdf(df, path, rows_per_page=40):
    # Use a custom page size (in mm) for a wider layout
    # Example: 500mm width × 297mm height (A3 height)
    pdf = FPDF(orientation='L', unit='mm', format=(500, 297))
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Arial", size=7)

    # Adjust column width: try to fit all columns in the page width
    page_width = pdf.w - 20  # Subtract margins
    col_width = page_width / max(len(df.columns), 1)

    # Print header
    for col in df.columns:
        pdf.cell(col_width, 10, str(col), border=1)
    pdf.ln()

    # Print rows
    for i, row in df.iterrows():
        if i % rows_per_page == 0 and i != 0:
            pdf.add_page()
            for col in df.columns:
                pdf.cell(col_width, 10, str(col), border=1)
            pdf.ln()

        for item in row:
            if pd.isna(item):
                item_str = ""
            elif isinstance(item, float):
                item_str = f"{item:.2f}"
            else:
                item_str = str(item)
            pdf.cell(col_width, 10, item_str, border=1)
        pdf.ln()


    pdf.output(str(path))


# Main logic
xpt_files = glob.glob(str(RAW_PATH / "*.XPT"))

for file in xpt_files:
    base_name = Path(file).stem.lower()
    print(f"Processing: {base_name}")

    df, meta = pyreadstat.read_xport(file)
    df_cleaned = clean_dataframe(df)

    csv_path = CLEANED_PATH / f"{base_name}_cleaned.csv"
    df_cleaned.to_csv(csv_path, index=False)

    pdf_path = CLEANED_PATH / f"{base_name}_cleaned.pdf"
    save_df_to_pdf(df_cleaned, pdf_path)

    print(f"Saved cleaned data to: {csv_path} and PDF to: {pdf_path}")


Processing: bpx_i


C:\Users\Keminda\AppData\Local\Temp\ipykernel_16476\2512513180.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("", inplace=True)


Saved cleaned data to: ..\data\cleaned\bpx_i_cleaned.csv and PDF to: ..\data\cleaned\bpx_i_cleaned.pdf
Processing: ghb_i


C:\Users\Keminda\AppData\Local\Temp\ipykernel_16476\2512513180.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("", inplace=True)


Saved cleaned data to: ..\data\cleaned\ghb_i_cleaned.csv and PDF to: ..\data\cleaned\ghb_i_cleaned.pdf
Processing: ins_i


C:\Users\Keminda\AppData\Local\Temp\ipykernel_16476\2512513180.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("", inplace=True)


Saved cleaned data to: ..\data\cleaned\ins_i_cleaned.csv and PDF to: ..\data\cleaned\ins_i_cleaned.pdf
Processing: ogtt_i


C:\Users\Keminda\AppData\Local\Temp\ipykernel_16476\2512513180.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("", inplace=True)


Saved cleaned data to: ..\data\cleaned\ogtt_i_cleaned.csv and PDF to: ..\data\cleaned\ogtt_i_cleaned.pdf
